# 🧠 Notebook 05: Chain-of-Thought Prompting

**Time:** 20 minutes  
**Goal:** Improve reasoning quality by making LLMs "think step-by-step"

## What is Chain-of-Thought (CoT)?

Chain-of-Thought prompting encourages LLMs to break down complex problems into intermediate steps, similar to how humans solve problems.

**Without CoT:**
Q: What is 15% of 240?
A: 36

**With CoT:**
Q: What is 15% of 240?
A: Let me work through this step by step:

Convert 15% to decimal: 15/100 = 0.15
Multiply: 0.15 × 240 = 36
Therefore, 15% of 240 is 36.


## Why Does CoT Work?

- **Breaks down complexity** - Tackles one step at a time
- **Reduces errors** - Can catch mistakes in reasoning
- **Improves accuracy** - Especially on math, logic, and multi-step problems
- **Makes reasoning transparent** - You can see where it went wrong
- **Enables verification** - Each step can be checked

## When to Use CoT

✅ **Use CoT for:**
- Math and calculations
- Multi-step reasoning
- Logic problems
- Planning and strategy
- Analysis requiring multiple steps
- Debugging and troubleshooting

❌ **Skip CoT for:**
- Simple factual questions
- Single-step tasks
- When speed matters more than accuracy
- Creative writing (unless planning plot)

Let's master it! 🚀

**Prerequisites:** Notebooks 02-04 completed

In [1]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 05: CHAIN-OF-THOUGHT PROMPTING")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

print()
print("✓ Ready to learn Chain-of-Thought!")
print()

NOTEBOOK 05: CHAIN-OF-THOUGHT PROMPTING

Configuration loaded: Path C

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5
✓ Ollama client initialized
  Available models: ['llama3.2:3b']
  Default model: llama3.2:3b

✓ Ready to learn Chain-of-Thought!



---
## 🧪 Experiment 1: Before and After CoT

Let's see the dramatic difference CoT makes on a reasoning problem.

In [2]:
# Without vs With CoT
print("=" * 60)
print("EXPERIMENT 1: The Power of Chain-of-Thought")
print("=" * 60)
print()

problem = """A farmer has 17 sheep. All but 9 die. How many sheep are left?"""

# Without CoT
print("❌ WITHOUT Chain-of-Thought:")
print("-" * 60)

prompt_no_cot = f"{problem}\n\nAnswer:"

response_no_cot = client.generate(
    prompt=prompt_no_cot,
    temperature=0.0,
    max_tokens=50
)

if "error" not in response_no_cot:
    print(f"Prompt: {prompt_no_cot}")
    print()
    print(f"Response: {response_no_cot['content']}")
    tracker.add_call(response_no_cot)
else:
    print(f"Error: {response_no_cot['error']}")

print()
print()

# With CoT
print("✅ WITH Chain-of-Thought:")
print("-" * 60)

prompt_with_cot = f"""{problem}

Let's think through this step-by-step:"""

response_with_cot = client.generate(
    prompt=prompt_with_cot,
    temperature=0.0,
    max_tokens=150
)

if "error" not in response_with_cot:
    print(f"Prompt: {prompt_with_cot}")
    print()
    print(f"Response: {response_with_cot['content']}")
    tracker.add_call(response_with_cot)
else:
    print(f"Error: {response_with_cot['error']}")

print()
print()

print("=" * 60)
print("💡 OBSERVATION")
print("=" * 60)
print()
print("Notice how CoT:")
print("  • Shows its reasoning process")
print("  • Is more likely to get the correct answer (9 sheep)")
print("  • Makes it easy to spot errors in logic")
print("  • Builds confidence in the answer")
print()

EXPERIMENT 1: The Power of Chain-of-Thought

❌ WITHOUT Chain-of-Thought:
------------------------------------------------------------
Prompt: A farmer has 17 sheep. All but 9 die. How many sheep are left?

Answer:

Response: To find the answer, we need to subtract the number of sheep that died (9) from the total number of sheep.

Total number of sheep = 17
Number of sheep that died = 9

Sheep left = Total number of sheep


✅ WITH Chain-of-Thought:
------------------------------------------------------------
Prompt: A farmer has 17 sheep. All but 9 die. How many sheep are left?

Let's think through this step-by-step:

Response: To find the number of sheep left, we need to subtract the number of sheep that died from the total number of sheep.

First, let's identify the information given:

* Total number of sheep = 17
* Number of sheep that died = 9

Now, let's perform the subtraction:

Number of sheep left = Total number of sheep - Number of sheep that died
= 17 - 9
= 8

So, there are 8 

---
## 🎯 CoT Prompting Patterns

There are several ways to trigger Chain-of-Thought reasoning:

### Pattern 1: "Let's think step-by-step"

**Most common and effective:**
```
Problem: [your problem]

Let's think step-by-step:
```

### Pattern 2: "Let's work through this together"

**More conversational:**
```
Problem: [your problem]

Let's work through this together:
1.
```

### Pattern 3: "Before answering, show your work"

**Explicit instruction:**
```
Problem: [your problem]

Before providing your final answer, show all your work and reasoning:
```

### Pattern 4: "First, let's understand... Then..."

**Structured approach:**
```
Problem: [your problem]

First, let's understand what we're being asked.
Then, let's break it down into steps.
Finally, let's solve it.
```

In [3]:
# Testing Different CoT Patterns
print("=" * 60)
print("EXPERIMENT 2: Different CoT Patterns")
print("=" * 60)
print()

problem = """If a train travels 120 miles in 2 hours, then 180 miles in the next 3 hours, 
what is the average speed for the entire journey?"""

patterns = {
    "Pattern 1: Step-by-step": f"{problem}\n\nLet's think step-by-step:",
    
    "Pattern 2: Work together": f"{problem}\n\nLet's work through this together:",
    
    "Pattern 3: Show work": f"{problem}\n\nBefore providing your final answer, show all your work and reasoning:",
    
    "Pattern 4: Structured": f"""{problem}

First, let's understand what we're being asked.
Then, let's break it down into steps.
Finally, let's calculate the answer."""
}

for pattern_name, prompt in patterns.items():
    print(f"🎯 {pattern_name}")
    print("-" * 60)
    
    response = client.generate(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )
    
    if "error" not in response:
        print(response['content'][:300] + "..." if len(response['content']) > 300 else response['content'])
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("💡 All patterns work! Choose based on your preference and use case.")
print()

EXPERIMENT 2: Different CoT Patterns

🎯 Pattern 1: Step-by-step
------------------------------------------------------------
To find the average speed for the entire journey, we need to calculate the total distance traveled and the total time taken.

Step 1: Calculate the total distance traveled
Total distance = Distance traveled in first part of journey + Distance traveled in second part of journey
= 120 miles + 180 mile...


🎯 Pattern 2: Work together
------------------------------------------------------------
To find the average speed for the entire journey, we need to calculate the total distance traveled and the total time taken.

Total distance = Distance traveled in first part + Distance traveled in second part
= 120 miles + 180 miles
= 300 miles

Total time = Time taken for first part + Time taken f...


🎯 Pattern 3: Show work
------------------------------------------------------------
To find the average speed for the entire journey, we need to first calculate the total dist

---
## 🔢 Math and Calculations

CoT shines brightest on math problems. Let's test it!

In [4]:
# CoT for Math Problems
print("=" * 60)
print("EXPERIMENT 3: CoT for Math Problems")
print("=" * 60)
print()

math_problems = [
    "If apples cost $3 per pound and oranges cost $2 per pound, how much would 4 pounds of apples and 6 pounds of oranges cost?",
    
    "A store offers a 20% discount on items over $50, and then an additional $10 off. How much would a $80 item cost after both discounts?",
    
    "If a car uses 1 gallon of gas every 25 miles, how many gallons are needed for a 340-mile trip?"
]

for i, problem in enumerate(math_problems, 1):
    print(f"Problem {i}:")
    print("-" * 60)
    print(problem)
    print()
    
    prompt = f"{problem}\n\nLet's solve this step-by-step:"
    
    response = client.generate(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )
    
    if "error" not in response:
        print("Solution:")
        print(response['content'])
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("💡 CoT makes math problems much more reliable!")
print()

EXPERIMENT 3: CoT for Math Problems

Problem 1:
------------------------------------------------------------
If apples cost $3 per pound and oranges cost $2 per pound, how much would 4 pounds of apples and 6 pounds of oranges cost?

Solution:
To find the total cost, we need to calculate the cost of each type of fruit separately and then add them together.

Step 1: Calculate the cost of 4 pounds of apples
Cost per pound of apples = $3
Weight of apples = 4 pounds
Total cost of apples = Cost per pound x Weight of apples
= $3 x 4
= $12

Step 2: Calculate the cost of 6 pounds of oranges
Cost per pound of oranges = $2
Weight of oranges = 6 pounds
Total cost of oranges = Cost per pound x Weight of oranges
= $2 x 6
= $12

Step 3: Add the total cost of apples and oranges together
Total cost = Total cost of apples + Total cost of oranges
= $12 + $12
= $24

Therefore, 4 pounds of apples and 6 pounds of oranges would cost $24.


Problem 2:
----------------------------------------------------------

---
## 🧩 Logic and Reasoning

CoT is excellent for logic puzzles and multi-step reasoning.

In [5]:
# CoT for Logic Problems
print("=" * 60)
print("EXPERIMENT 4: CoT for Logic Problems")
print("=" * 60)
print()

logic_problem = """
Three people (Alice, Bob, and Carol) are standing in a line.
- Alice is not first
- Bob is not last
- Carol is not in the middle

What is the order from first to last?
"""

print("Logic Problem:")
print(logic_problem)
print()

# Without CoT
print("WITHOUT CoT:")
print("-" * 60)

response_no_cot = client.generate(
    prompt=f"{logic_problem}\n\nAnswer:",
    temperature=0.0,
    max_tokens=50
)

if "error" not in response_no_cot:
    print(response_no_cot['content'])
    tracker.add_call(response_no_cot)

print()
print()

# With CoT
print("WITH CoT:")
print("-" * 60)

cot_prompt = f"""{logic_problem}

Let's think through this step-by-step:
1. First, let's list what we know
2. Then, let's eliminate impossible arrangements
3. Finally, let's determine the correct order"""

response_with_cot = client.generate(
    prompt=cot_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_with_cot:
    print(response_with_cot['content'])
    tracker.add_call(response_with_cot)

print()
print("💡 CoT helps break down complex constraints into manageable steps!")
print()

EXPERIMENT 4: CoT for Logic Problems

Logic Problem:

Three people (Alice, Bob, and Carol) are standing in a line.
- Alice is not first
- Bob is not last
- Carol is not in the middle

What is the order from first to last?


WITHOUT CoT:
------------------------------------------------------------
Based on the given conditions:

* Alice is not first
* Bob is not last
* Carol is not in the middle

The only possible position for Carol is either at the beginning or end of the line. Since Alice cannot be first, and


WITH CoT:
------------------------------------------------------------
Let's go through the steps:

**Step 1: List what we know**

- Alice is not first
- Bob is not last
- Carol is not in the middle (meaning she must be either first or last)

So, we have three pieces of information to work with:

1. A cannot be first
2. B cannot be last
3. C is in an extreme position (first or last)

**Step 2: Eliminate impossible arrangements**

Since Carol can't be in the middle, she must be 

---
## 🔗 Multi-Step Planning

CoT is perfect for planning tasks that require multiple sequential steps.

In [6]:
# CoT for Planning
print("=" * 60)
print("EXPERIMENT 5: CoT for Multi-Step Planning")
print("=" * 60)
print()

planning_task = """
You need to organize a small team meeting for 8 people next Tuesday at 2 PM.
You need to: book a room, send invitations, prepare an agenda, and order snacks.
The room needs AV equipment. Some team members are remote.

What should you do first, and in what order should you complete these tasks?
"""

print("Planning Task:")
print(planning_task)
print()

cot_planning_prompt = f"""{planning_task}

Let's plan this systematically:
1. First, let's identify all the dependencies
2. Then, let's determine what must be done first
3. Finally, let's create the optimal sequence"""

response = client.generate(
    prompt=cot_planning_prompt,
    temperature=0.3,
    max_tokens=400
)

if "error" not in response:
    print("CoT Planning Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
else:
    print(f"Error: {response['error']}")

print()
print("💡 CoT helps identify dependencies and create logical sequences!")
print()

EXPERIMENT 5: CoT for Multi-Step Planning

Planning Task:

You need to organize a small team meeting for 8 people next Tuesday at 2 PM.
You need to: book a room, send invitations, prepare an agenda, and order snacks.
The room needs AV equipment. Some team members are remote.

What should you do first, and in what order should you complete these tasks?


CoT Planning Response:
Let's break down the tasks into smaller steps and identify the dependencies:

**Task Dependencies:**

1. Booking a room → requires availability of rooms (no dependency)
2. Sending invitations → depends on booking a room
3. Preparing an agenda → can be done independently before or after sending invitations
4. Ordering snacks → can be done independently at any time

The remote team members do not affect the sequence of tasks, as they are already part of the meeting setup.

**Task Order:**

Based on the dependencies, here's a suggested order:

1. **Book a room**: Start by checking availability and booking a suitable 

---
## 🎯 Your Turn: Practice Tasks

Time to practice using Chain-of-Thought prompting!

### 📝 Task 1: Solve a Multi-Step Problem

**Goal:** Use CoT to solve a problem that requires multiple reasoning steps.

In [ ]:
# TODO - Task 1: Multi-Step Problem Solving
print("=" * 60)
print("TASK 1: Multi-Step Problem Solving")
print("=" * 60)
print()

# ============================================================================
# TODO: Create your own multi-step problem OR use one of these examples:
# ============================================================================

your_problem = """
A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?
"""

# ============================================================================
# TODO: Write your CoT prompt
# ============================================================================

your_cot_prompt = f"""
Here is the problem {your_problem}
Let's think step by step:
1. Identify the cost of the first hour.
2. Calculate how many additional hours there are.
3. Compute the total parking cost before applying the daily maximum.
4. Compare that total with the daily maximum charge.
5. Give the final answer clearly.
"""

print("Your Problem:")
print("-" * 60)
print(your_problem)
print()

print("Your CoT Prompt:")
print("-" * 60)
print(your_cot_prompt)
print()

# Test WITHOUT CoT first
print("TEST 1: Without CoT (for comparison)")
print("=" * 60)

response_without = client.generate(
    prompt=f"{your_problem}\n\nAnswer:",
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_without:
    print(response_without['content'])
    tracker.add_call(response_without)
    without_answer = response_without['content']
else:
    without_answer = "Error occurred"

print()
print()

# Test WITH CoT
print("TEST 2: With CoT")
print("=" * 60)

response_with = client.generate(
    prompt=your_cot_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_with:
    print(response_with['content'])
    tracker.add_call(response_with)
    with_answer = response_with['content']
else:
    with_answer = "Error occurred"

print()
print()

# ========================================================================
# TODO: Analyze the results
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Problem Chosen
A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?

### Without CoT Analysis

**Answer received:** {without_answer[:100]}...

**Correct?** No

It fails to consider the maximum daily charge.

### With CoT Analysis

**Reasoning shown:** 

Get first hour charge, add extra hours charges, then the total charege, compare it to mmaximum and use the lower of the two.

**Final answer:** 25

**Correct?** Yes

### Comparison

**Which approach was more accurate?** With CoT

**Which was more transparent?** Both

**Which gave you more confidence?** With CoT

### Quality of Reasoning

**Did CoT show all necessary steps?** Yes

All required steps are showing.

**Could you verify each step?** Yes
All steps are correct 2 out of 3 times.

### Key Insight

When CoT helps most? Mutli-step reasonsing

### Real-World Application
The could be applied to all senarios required mutli-step reasoning 
"""

print(reflection)

append_to_reflection(
    notebook="05",
    section_title="Task 1 - Multi-Step Problem Solving",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 1: Multi-Step Problem Solving

Your Problem:
------------------------------------------------------------

A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?


Your CoT Prompt:
------------------------------------------------------------

Here is the problem 
A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?

Let's think step by step:
1. Identify the cost of the first hour.
2. Calculate how many additional hours there are.
3. Compute the total parking cost before applying the daily maximum.
4. Compare that total with the daily maximum charge.
5. Give the final answer clearly.


TEST 1: Without CoT (for comparison)
To find out how much you pay, let's break it down:

1. The first hour costs $5.
2. For the remaining 8 hours (since you parked for 9 hours), the rate 

### 📝 Task 2: Debug Faulty Reasoning

**Goal:** Use CoT to identify where reasoning goes wrong.

**Scenario:** Sometimes LLMs make mistakes. CoT makes these visible!

In [14]:
# TODO - Task 2: Debug Faulty Reasoning
print("=" * 60)
print("TASK 2: Debug Faulty Reasoning")
print("=" * 60)
print()

# A problem designed to potentially trip up the LLM
tricky_problem = """
A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?
"""

print("Tricky Problem:")
print(tricky_problem)
print()
print("(Common wrong answer: $0.10)")
print("(Correct answer: $0.05)")
print()

# ============================================================================
# TODO: Craft a CoT prompt that helps the LLM avoid the trap
# ============================================================================

debugging_prompt = f"""
# YOUR DEBUGGING PROMPT HERE

Example:
{tricky_problem}

Let's think very carefully step-by-step:
1. First, let's define our variables
2. Then, let's write out the equations
3. Then, let's solve systematically
4. Finally, let's verify our answer
"""

response = client.generate(
    prompt=debugging_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("CoT Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
    
    # ========================================================================
    # TODO: Analyze if it got the right answer
    # ========================================================================
    
    print()
    print("=" * 60)
    print("REFLECTION")
    print("=" * 60)
    print()
    
    reflection = """
### Did it get the correct answer? ($0.05)

Yes

### Reasoning Quality

Yes
Step 1: Correct: Define Variables T and B
Step 2: Correct: Write Out Equations B + T = 1.10 and T = B + 1.00
Step 3: Correct: Substitute equation (2) into equation (1) we get B + (B + 1.00) = 1.10 then B = 0.05


### If it got it wrong:

**Where did the reasoning fail?**

N/A

**How would you fix the prompt to help it?**

N/A

### If it got it right:

**What made the CoT effective?**

The question is broken down and each step is ensured to be correct.

**Could you simplify the prompt and still get it right?**

Yes, but the same principle applies.

### Key Learning

**What makes a problem "tricky"?**

It tricks the people or the model to get the fast intuitive reponse which is wrong after step-by-step verification.

**How does CoT help with tricky problems?**

It enfore the verification of each step.

### Debugging Strategy

**How would you use CoT to debug LLM reasoning in production?**
I will require the model to show/implement step-by-step reasonsing process to verify/ensure reliability.  
"""
    
    print(reflection)
    
    append_to_reflection(
        notebook="05",
        section_title="Task 2 - Debug Faulty Reasoning",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")

else:
    print(f"Error: {response['error']}")

print()

TASK 2: Debug Faulty Reasoning

Tricky Problem:

A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?


(Common wrong answer: $0.10)
(Correct answer: $0.05)

CoT Response:
# Bat and Ball Cost Problem Debugging Prompt

A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?

## Step 1: Define Variables
Let's define two variables to represent the unknowns:
- Let B be the cost of the ball in dollars.
- Let T be the cost of the bat in dollars.

We know that the bat costs $1.00 more than the ball, so we can write an equation based on this information:

T = B + 1

## Step 2: Write Out Equations
From the problem statement, we also know that together the bat and the ball cost $1.10:
B + T = 1.10

Substituting the expression for T from the first step into this equation gives us an equation in terms of B only:
B + (B + 1) = 1.10

## Step 3: Solve Systematically
To solve this equation, we'll c

### 📝 Task 3: Compare CoT Across Different Models/Temperatures

**Goal:** Understand how CoT interacts with model settings.

**Experiment:** Test how temperature affects CoT quality.

In [ ]:
# TODO - Task 3: CoT with Different Settings
print("=" * 60)
print("TASK 3: CoT Across Different Settings")
print("=" * 60)
print()

test_problem = """
A store has a sale: "Buy 2, get 1 free" on items that cost $15 each.
If you want to get 7 items, how much will you pay?
"""

print("Problem:")
print(test_problem)
print()

cot_prompt_base = f"""{test_problem}

Let's work through this step-by-step:
"""

# Test different temperatures
temperatures = [0.0, 0.5, 1.0]

results = {}

for temp in temperatures:
    print(f"Testing with temperature={temp}")
    print("-" * 60)
    
    response = client.generate(
        prompt=cot_prompt_base,
        temperature=temp,
        max_tokens=600
    )
    
    if "error" not in response:
        print(response['content'][:1000] + "..." if len(response['content']) > 1000 else response['content'])
        results[temp] = response['content']
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
        results[temp] = "Error"
    
    print()
    print()
    time.sleep(0.5)

# ========================================================================
# TODO: Analyze the differences
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Temperature Comparison

#### Temperature 0.0 (Deterministic)

**Reasoning quality:** ___2/5

**Final answer:** 30

**Consistency:** No

**Observations:**
The model mistakes the price of set for item; besides it can't get the correct price of the 7th item believing it's free.
Besides, it is not consistent

#### Temperature 0.5 (Balanced)

**Reasoning quality:** __5_/5

**Final answer:** 75

**Different from temp 0.0?** Yes, it is correct.

**Observations:**
I also shows randomness, the result after reunning is wrong.

#### Temperature 1.0 (Creative)

**Reasoning quality:** _3__/5

**Final answer:** 60

**Different from others?** Yes, it gets the first two sets correct but think the 7th item is free.

**Observations:**
The first run's result is logical except for the 7th item 

### Overall Analysis

**Best temperature for CoT reasoning:** 0.5

**Why?**
The balance may make it seem right, but I would still expect 0.0 to have a higher chance of giving the correct result.
None of the three is consistent, so it feels pointless to evaluate them.

**Does CoT work better with lower temperatures?**
Yes, maybe that's the reason why 0.5 got it right (once)

### Trade-offs

**Accuracy vs Creativity:**
The lower the temperature, the more accurate but less creative the model tends to be.

**When might you want higher temperature with CoT?**
For brainstorming tasks that still require logical thinking, for example brainstorming possible strategy changes.

**When should you always use temperature 0.0?**
For tasks that require high accuracy, such as accounting calculations.

### Production Recommendations

**For math/logic problems:** Temperature = _0.0__

**For planning/strategy:** Temperature = __0.5_

**For creative problem-solving:** Temperature = __0.8_

**Reasoning:** 
Lower temperature ensures stable and correct outputs for deterministic tasks, while moderate temperature allows some flexibility for planning without losing coherence. 
Higher temperature is best for creativity and idea generation, where variation is more important than strict correctness.
"""

print(reflection)

append_to_reflection(
    notebook="05",
    section_title="Task 3 - CoT with Different Settings",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: CoT Across Different Settings

Problem:

A store has a sale: "Buy 2, get 1 free" on items that cost $15 each.
If you want to get 7 items, how much will you pay?


Testing with temperature=0.0
------------------------------------------------------------
To find out how much we'll pay, let's break it down step by step:

1. The sale offers "Buy 2, get 1 free". This means that for every 3 items, we only pay for 2.

2. We want to buy 7 items.

3. To make the most of the deal, we can divide 7 by 3, since it's the smallest unit of purchase with the sale. 

   Let's do this calculation:
   7 ÷ 3 = 2 (with a remainder of 1)

4. This means that for every 3 items, we'll pay for 2 and get 1 free. We have 2 groups of 3 items.

5. Each item costs $15. Since we're paying for 2 items in each group, the cost per group is:
   2 x $15 = $30

6. To find out how much we'll pay in total, multiply the cost per group by the number of groups:
   $30 x 2 = $60

7. We got a remainder of 1 item (the last 

---
## 🎓 Advanced CoT Techniques

In [26]:
# Advanced CoT Patterns
print("=" * 60)
print("ADVANCED COT TECHNIQUES")
print("=" * 60)
print()

advanced_techniques = {
    "Self-Consistency": """
    Run the same CoT prompt multiple times (with temp > 0) and take the majority answer.
    
    Example:
    - Run 5 times
    - Get answers: [42, 42, 43, 42, 42]
    - Final answer: 42 (appears 4/5 times)
    
    Use when: High stakes decisions, uncertain problems
    """,
    
    "Least-to-Most": """
    Break complex problems into simpler sub-problems, solve them in order.
    
    Example:
    "Let's start with the simplest part:
     1. First, solve for X
     2. Using X, now solve for Y
     3. Finally, combine X and Y to get Z"
    
    Use when: Hierarchical problems, building up complexity
    """,
    
    "Tree of Thoughts": """
    Explore multiple reasoning paths, then choose the best.
    
    Example:
    "Let's explore two approaches:
     Approach 1: [reasoning path 1]
     Approach 2: [reasoning path 2]
     
     Now let's evaluate which is better..."
    
    Use when: Multiple valid approaches, complex trade-offs
    """,
    
    "Verification": """
    Add explicit verification steps.
    
    Example:
    "Let's solve step-by-step:
     [steps]
     
     Now let's verify our answer:
     [check each step]
     [plug back into original problem]"
    
    Use when: Critical applications, error-prone calculations
    """,
    
    "Reflection": """
    Ask the model to reflect on its own reasoning.
    
    Example:
    "After showing your reasoning:
     1. Identify any assumptions you made
     2. Consider if there are errors
     3. Suggest improvements to your approach"
    
    Use when: Learning, improvement, critical analysis
    """
}

for technique, description in advanced_techniques.items():
    print(f"🎯 {technique}")
    print("-" * 60)
    print(description)
    print()

print()
print("💡 These advanced techniques can significantly improve reasoning quality!")
print("   Experiment with them in your projects.")
print()

ADVANCED COT TECHNIQUES

🎯 Self-Consistency
------------------------------------------------------------

    Run the same CoT prompt multiple times (with temp > 0) and take the majority answer.

    Example:
    - Run 5 times
    - Get answers: [42, 42, 43, 42, 42]
    - Final answer: 42 (appears 4/5 times)

    Use when: High stakes decisions, uncertain problems
    

🎯 Least-to-Most
------------------------------------------------------------

    Break complex problems into simpler sub-problems, solve them in order.

    Example:
    "Let's start with the simplest part:
     1. First, solve for X
     2. Using X, now solve for Y
     3. Finally, combine X and Y to get Z"

    Use when: Hierarchical problems, building up complexity
    

🎯 Tree of Thoughts
------------------------------------------------------------

    Explore multiple reasoning paths, then choose the best.

    Example:
    "Let's explore two approaches:
     Approach 1: [reasoning path 1]
     Approach 2: [reaso

---
## 📊 CoT Best Practices

In [28]:
# CoT Best Practices
print("=" * 60)
print("CHAIN-OF-THOUGHT BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "When to Use CoT": [
        "✓ Multi-step problems (math, logic, planning)",
        "✓ When accuracy is critical",
        "✓ When you need to verify reasoning",
        "✓ When the problem is complex or tricky",
        "✓ When debugging incorrect answers",
        "✗ Simple factual questions",
        "✗ Single-step tasks",
        "✗ When speed is more important than accuracy"
    ],
    
    "Prompting Techniques": [
        "✓ Use explicit phrases: 'Let's think step-by-step'",
        "✓ Guide the structure: numbered steps, categories",
        "✓ Ask for verification: 'Now check your work'",
        "✓ Request final answer separately: 'Therefore, the answer is...'",
        "✓ Use temperature 0.0 for consistency",
        "✗ Don't assume CoT will happen automatically",
        "✗ Don't leave reasoning structure vague"
    ],
    
    "Parsing CoT Outputs": [
        "✓ Look for clear step markers (1., 2., 3.)",
        "✓ Extract final answer explicitly",
        "✓ Verify each step if critical",
        "✓ Store reasoning for debugging",
        "✓ Use structured formats when possible",
        "✗ Don't just grab the last number",
        "✗ Don't skip verification on critical tasks"
    ],
    
    "Production Use": [
        "✓ Cache successful CoT patterns",
        "✓ Monitor reasoning quality over time",
        "✓ A/B test with and without CoT",
        "✓ Combine with structured output for final answer",
        "✓ Implement verification for high-stakes decisions",
        "✗ Don't use CoT everywhere (cost/latency)",
        "✗ Don't trust reasoning blindly"
    ]
}

for category, practices in best_practices.items():
    print(f"📌 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES FOR COT")
print("=" * 60)
print()
print("1. CoT works best with temperature 0.0")
print("2. Always explicitly request step-by-step reasoning")
print("3. Verify critical outputs - don't trust blindly")
print("4. Use CoT for accuracy, skip for speed")
print("5. Combine CoT with structured output for best results")
print()

CHAIN-OF-THOUGHT BEST PRACTICES

📌 When to Use CoT
------------------------------------------------------------
  ✓ Multi-step problems (math, logic, planning)
  ✓ When accuracy is critical
  ✓ When you need to verify reasoning
  ✓ When the problem is complex or tricky
  ✓ When debugging incorrect answers
  ✗ Simple factual questions
  ✗ Single-step tasks
  ✗ When speed is more important than accuracy

📌 Prompting Techniques
------------------------------------------------------------
  ✓ Use explicit phrases: 'Let's think step-by-step'
  ✓ Guide the structure: numbered steps, categories
  ✓ Ask for verification: 'Now check your work'
  ✓ Request final answer separately: 'Therefore, the answer is...'
  ✓ Use temperature 0.0 for consistency
  ✗ Don't assume CoT will happen automatically
  ✗ Don't leave reasoning structure vague

📌 Parsing CoT Outputs
------------------------------------------------------------
  ✓ Look for clear step markers (1., 2., 3.)
  ✓ Extract final answer explici

---
## ✅ Notebook 05 Complete!

### Summary

You've mastered Chain-of-Thought prompting! You now know:
- ✅ What CoT is and why it works
- ✅ Multiple CoT prompting patterns
- ✅ When to use (and not use) CoT
- ✅ How to debug reasoning with CoT
- ✅ Advanced CoT techniques
- ✅ Production best practices

In [29]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on Chain-of-Thought
# ============================================================================

reflection = """
### 1. How has CoT changed your approach to complex problems?

It makes my approach more structured. Instead of asking the question directly, I now break it into several steps.

### 2. When will you use CoT vs regular prompting?

CoT is better for complex, multi-step problems that require reasoning.
Regular prompting is better for simple tasks or factual checks.

### 3. What's your confidence in using CoT? (1-5)

**Confidence:** __3/5

More practice and more problem-solving experience on real-world projects would increase my confidence.

### 4. Most surprising finding about CoT?

With CoT, temperature 0.5 gave the correct answer while 0.0 was wrong.

### 5. How will you use CoT in your projects?
Implement it in workflows that require multi-step reasoning (e.g.architecture-level projects) to reduce the chance of errors.

### 6. CoT limitations you discovered?

The results can still be inconsistent: model may produce different answers after rerunning the same prompt.
Also, if the steps in the prompt are wrong, the output will likely be wrong as well.

### 7. Key takeaway from this notebook?
CoT is a powerful tool for improving reasoning. Using it wisely can help avoid unnecessary setbacks.

"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="05",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 05 COMPLETE!")
print("=" * 60)
print()
print("Progress: [████████████████░░░░] 63% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs")
print("✓ Notebook 05: Chain of Thought ← YOU ARE HERE")
print("○ Notebook 06: Model Comparison")
print("○ Notebook 07: MCP Introduction")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/06_model_comparison.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. How has CoT changed your approach to complex problems?

It makes my approach more structured. Instead of asking the question directly, I now break it into several steps.

### 2. When will you use CoT vs regular prompting?

CoT is better for complex, multi-step problems that require reasoning.
Regular prompting is better for simple tasks or factual checks.

### 3. What's your confidence in using CoT? (1-5)

**Confidence:** __3/5

More practice and more problem-solving experience on real-world projects would increase my confidence.

### 4. Most surprising finding about CoT?

With CoT, temperature 0.5 gave the correct answer while 0.0 was wrong.

### 5. How will you use CoT in your projects?
Implement it in workflows that require multi-step reasoning (e.g.architecture-level projects) to reduce the chance of errors.

### 6. CoT limitations you discovered?

The results can still be inconsistent: model may produce different answers after rerunning the sam